# Case Centric Indexer
Similiar to gene-centric, but with case and gene reversed
```
case{}
     |___ gene[]
             |___ ssm[]
                   |___ consequence[]
                   |             |_____ transcript{}
                   |                          |_____ annotation{}
                   |___ observation[]
```

In [1]:
import os
import requests
import uuid
%load_ext autoreload
from exports.mappings import GeneMapper, SSMMapper, Mapper
from exports.utils import get_array_paths

from pyspark.sql.functions import col, explode, collect_list, size, sum, first, struct, udf, regexp_extract, lit, count, broadcast
from pyspark.sql.types import StringType

## Load combined maf into spark

In [2]:
url = 's3a://test/small_combined_mafs.csv'
    
df = sqlContext.read.format('com.databricks.spark.csv')\
                .options(header='true', inferschema='true')\
                .load(url)\
                .drop_duplicates()

In [3]:
#df = df.limit(500000)

In [4]:
'''
df = sqlContext.read.format('com.databricks.spark.csv')\
                .options(header='true', inferschema='true', comment='#', delimiter='\t')\
                .load('/home/ubuntu/tests/data/test.maf')
'''        

"\ndf = sqlContext.read.format('com.databricks.spark.csv')                .options(header='true', inferschema='true', comment='#', delimiter='\t')                .load('/home/ubuntu/tests/data/test.maf')\n"

## Rename and select desired columns in the mafs

In [4]:
%autoreload
from exports.utils import (
    maf_annotation_map,
    maf_gene_map,
    maf_observation_map,
    maf_ssm_map,
    maf_transcript_map,
    tumor_genotype_map,
    tumor_validation_map,
    normal_genotype_map,
    sample_map,
    input_bam_map,
    read_depth_map,
    maf_cols
)

maf_df = df.select(*( col(v).alias(k) for k,v in maf_cols.items() ))

## Augment maf df by extracting submitter_id and creating ssm_uuids

In [5]:
maf_df = maf_df.withColumn('_case_submitter_id',
                           regexp_extract(col('tumor_sample_barcode'),
                                          '([A-Z]{4}-[A-Z0-9]{2}-[A-Z0-9]{4})',1))
maf_ssm_map.update({'_case_submitter_id':'_case_submitter_id'})

In [6]:
def ssm_uuid(chromosome, start_position, ref_allele, tumor_allele):
    '''
    SNP: "{chromosome}:g.{start_position}{reference_allele}>{tumor_allele}"
    DEL: "{chromosome}:g.{start_position}del{reference_allele}"
    INS: "{chromosome}:g.{start_position}_{end_position}ins{tumor_allele}"
    '''
    chromosome = chromosome.replace('chr','')
    label = '{}:g.{}:{}>{}'.format(chromosome, start_position, ref_allele, tumor_allele)
    return str(uuid.uuid5(uuid.UUID('d15296a3-38ed-412e-8ace-75e235f82f55'), label))

ssm_uuid_udf =udf(ssm_uuid, StringType())
maf_df = maf_df.withColumn('ssm_uuid', ssm_uuid_udf(col('chromosome'), col('start_position'), col('reference_allele'), col('tumor_allele')))
maf_observation_map.update({'ssm_uuid':'ssm_uuid'})
maf_ssm_map.update({'ssm_uuid':'ssm_uuid'})

## Slice and dice until we get to the format we want

### Gene df

In [7]:
gene_df = maf_df.select(*( col(k) for k in maf_gene_map.keys() + ['_case_submitter_id'] ))
# Fill in empty data we don't know about
gene_df = gene_df.withColumn('description', lit(None).cast(StringType()))\
                 .drop_duplicates()

In [8]:
#gene_df.count()

### SSM df

In [9]:
ssm_df = maf_df.select(*( col(k) for k in maf_ssm_map.keys() ))\
               .drop_duplicates()

### Transcript-annotation df

```
transcript{}
     |_____ annotation{}
```

In [10]:
# Rename columns
tran_anno_df = maf_df.select(*( maf_transcript_map.keys() + maf_annotation_map.keys() ))
# Select annotation into nested format
tran_anno_df = tran_anno_df.select(struct(*maf_annotation_map.keys()).alias('annotation'), *maf_transcript_map.keys())\
                           .drop_duplicates()
#tran_anno_df.printSchema()

In [11]:
#tran_anno_df.count()

### Observation df

In [12]:
observation_df = maf_df.select(*(maf_observation_map.keys()
                                 +normal_genotype_map.keys()
                                 +tumor_genotype_map.keys()
                                 +tumor_validation_map.keys()
                                 +read_depth_map.keys()
                                 +input_bam_map.keys()
                                 +sample_map.keys()))

observation_df = observation_df.select(struct(*normal_genotype_map.keys()).alias('normal_genotype'),
                                       struct(*tumor_genotype_map.keys()).alias('tumor_genotype'),
                                       struct(*tumor_validation_map.keys()).alias('validation'),
                                       struct(*read_depth_map.keys()).alias('read_depth'),
                                       struct(*input_bam_map.keys()).alias('input_bam_file'),
                                       struct(*sample_map.keys()).alias('sample'),
                                       *maf_observation_map.keys())\
                                .drop('ssm_uuid')\
                                .drop_duplicates()

### Get case dataframe from existing graph

In [13]:
'''
doc = requests.get('http://elasticsearchvis.service.consul:9200/gdc_from_graph/case/_search',
                   auth=(os.environ.get("GDC_ES_USER"),
                         os.environ.get("GDC_ES_PASS"))
                  ).json()['hits']['hits'][0]['_source']

docs = [ r['_source'] for r in requests.get('http://elasticsearch.service.consul:9200/gdc_from_graph/case/_search?size=1',
                  ).json()['hits']['hits']]

paths = [ get_array_paths(d) for d in docs ]
paths = reduce(set.union, map(set, paths))
'''

'\ndoc = requests.get(\'http://elasticsearchvis.service.consul:9200/gdc_from_graph/case/_search\',\n                   auth=(os.environ.get("GDC_ES_USER"),\n                         os.environ.get("GDC_ES_PASS"))\n                  ).json()[\'hits\'][\'hits\'][0][\'_source\']\n\ndocs = [ r[\'_source\'] for r in requests.get(\'http://elasticsearch.service.consul:9200/gdc_from_graph/case/_search?size=1\',\n                  ).json()[\'hits\'][\'hits\']]\n\npaths = [ get_array_paths(d) for d in docs ]\npaths = reduce(set.union, map(set, paths))\n'

In [14]:
#doc = requests.get('http://elasticsearch.service.consul:9200/gdc_from_graph_35/_search').json()['hits']['hits'][0]['_source']
case_df = sqlContext.read.format("es")\
    .option('es.nodes', 'elasticsearch.service.consul')\
    .option('es.read.field.include', 'case_id,submitter_id,state,project.*,program.*,exposures.*,demographic.*')\
    .option('es.read.field.as.array.include','')\
    .option('es.resource.read', 'gdc_from_graph/case')\
    .option('es.nodes.resolve.hostname','false')\
    .load("gdc_from_graph")

In [15]:
#case_df = case_df.sample(False, 0.01)
#case_df.printSchema()

In [16]:
#case_df.first()

In [17]:
case_df.columns

['case_id', 'demographic', 'exposures', 'project', 'state', 'submitter_id']

## Assemble constituent parts

### Merge annotation with transcript

In [18]:
cons_tran_anno_df = tran_anno_df.select(struct(struct(*tran_anno_df.columns).alias('transcript')).alias('consequence'), 'gene_symbol')\
                                .groupBy('gene_symbol')\
                                .agg(collect_list('consequence').alias('consequence'))
                                #.drop('gene_symbol')

In [19]:
#cons_tran_anno_df.count()

### Join observation with consequence

In [20]:
import random
def salt(key, doc_count=0):
    return str(random.randint(0,int(max(0,doc_count-1024)**5)))+key
salt_udf = udf(salt, StringType())

In [21]:
salted_observation_df = observation_df.withColumn('salt_key', salt_udf(col('gene_symbol')))\
                                     #.repartition(1024, 'salt_key')

In [22]:
salted_consequence_df = cons_tran_anno_df.withColumn('salt_key', salt_udf(col('gene_symbol')))\
                                         .repartition(64, 'salt_key')

In [23]:
cons_obs = salted_consequence_df.join(salted_observation_df, salted_consequence_df.gene_symbol == salted_observation_df.gene_symbol, 'outer')\
                            .drop(salted_consequence_df.gene_symbol)\
                            .drop(salted_consequence_df.salt_key)\
                            .select('gene_symbol','consequence',struct(*[c for c in salted_observation_df.columns if c != 'gene_symbol']).alias('observation'))                   

### Join consequence into ssm

In [24]:
# Salt consequence-observation
salted_cons_obs = cons_obs.withColumn('doc_count', size(col('consequence')))\
                            .withColumn('salt_key', salt_udf(col('gene_symbol'), col('doc_count')))

In [25]:
salted_ssm = ssm_df.withColumn('salt_key', salt_udf(col('gene_symbol')))
#salted_ssm.repartition('salt_key').count()

In [26]:
ssm_cons = salted_cons_obs.join(salted_ssm, salted_ssm.gene_symbol == salted_cons_obs.gene_symbol, 'left')\
                        .drop(salted_cons_obs.gene_symbol)\
                        .drop(salted_cons_obs.salt_key)\
                        .select('gene_symbol', struct('consequence','observation',*ssm_df.columns).alias('ssm'))\
                        .groupBy('gene_symbol')\
                        .agg(collect_list('ssm').alias('ssm'))

### Join ssm with gene

In [27]:
gene_ssm = gene_df.join(ssm_cons, gene_df.symbol == ssm_cons.gene_symbol, 'left')\
                    .drop(ssm_cons.gene_symbol)\
                    .select('_case_submitter_id', struct('ssm', *gene_df.columns).alias('gene'))

In [28]:
#gene_ssm.printSchema()

### Join case with gene

In [29]:
gene_df.count()

18428

In [ ]:
case_centric = case_df.join(gene_ssm, case_df.submitter_id == gene_ssm._case_submitter_id, 'left')\
                        .drop(gene_ssm._case_submitter_id)\
                        .groupBy(*case_df.columns)\
                        .agg(collect_list('gene').alias('gene'))

In [ ]:
case_centric.count()

## Export df to es

In [ ]:
sqlContext.sql("set spark.sql.shuffle.partitions=2048")

#### Graph es index

In [35]:
#print requests.get('http://elasticsearch.service.consul:9200/_cat/indices?v').text

#### New vis index

In [36]:
print requests.get('http://elasticsearchvis.service.consul:9200/_cat/indices?v').text

health status index                           uuid                   pri rep docs.count docs.deleted store.size pri.store.size
green  open   case                            RtC6hJevQd21JeOKNI1M-g  10   0      97052            0     20.3mb         20.3mb
green  open   .monitoring-kibana-2-2016.11.21 Gg2lRUkbSpOJ69-so0TSpg   1   1       4086            0        2mb            1mb
green  open   .monitoring-kibana-2-2016.11.20 sYEaQ1ukStibxpHDnPI0Zg   1   1      17188            0      7.6mb          3.8mb
green  open   .monitoring-data-2              pt6so5oJRiCd9z0JwoJyKQ   1   1         10            0     39.3kb         19.6kb
green  open   .monitoring-es-2-2016.11.19     cpecQtt3Tx-Y1HP0BP0tpQ   1   1     253250          666    324.6mb          162mb
green  open   .kibana                         VktMnBTYQzioA_in3JbLSg   1   1          6            0    605.2kb        302.6kb
green  open   .monitoring-kibana-2-2016.11.18 8Cxzih0JQzSEeCFLLBUgaw   1   1      16392            0      7.4mb

In [37]:
print requests.get('http://elasticsearchvis.service.consul:9200/_cat/count/case?v').text

epoch      timestamp count
1479706835 05:40:35  1000



In [38]:
%autoreload
from exports.mappings import CaseMapper
m = CaseMapper()

In [39]:
m.mapping['properties']['gene']['dynamic'] = 'true'
m.mapping['properties']['gene']['properties']['ssm']['dynamic'] = 'true'
m.mapping['properties']['gene']['properties']['ssm']['properties']['observation']['dynamic'] = 'true'
#m.mapping['properties']['case']['properties']['files']['properties']['cases']['dynamic'] = 'true'

In [48]:
import json

print requests.delete('http://elasticsearchvis.service.consul:9200/case').json()

data = json.dumps({"settings":{"index":{
                "refresh_interval":"2s",
                "number_of_shards":10,
                "number_of_replicas":0,
                "mapper.dynamic":False,
                "mapping.nested_fields.limit":100,
                "mapping.total_fields.limit":2000
            }},"mappings":{
                "case":m.mapping
            }})
#print requests.put('http://localhost:9200/test/', data=data).json()
print requests.put('http://elasticsearchvis.service.consul:9200/case', data=data).json()

{u'acknowledged': True}
{u'acknowledged': True, u'shards_acknowledged': True}


In [49]:
#%%time
case_centric.persist().coalesce(1024).write.format('org.elasticsearch.spark.sql')\
                    .option('es.nodes', 'elasticsearchvis.service.consul')\
                    .option('es.nodes.resolve.hostname','false')\
                    .option('es.resource.write', 'case/case')\
                    .option('es.http.timeout', '10m')\
                    .option('es.http.retries', '300')\
                    .option('es.batch.write.retry.count', '100')\
                    .option('es.batch.write.retry.wait', '10m')\
                    .option('es.batch.size.bytes','100mb')\
                    .option('es.batch.size.entries', '10000')\
                    .option('es.batch.write.refresh ', 'true')\
                    .save('case/case')

Py4JJavaError: An error occurred while calling o812.save.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 74 in stage 78.0 failed 4 times, most recent failure: Lost task 74.3 in stage 78.0 (TID 39189, 172.21.30.228): org.apache.spark.util.TaskCompletionListenerException: Connection error (check network and/or proxy settings)- all nodes failed; tried [[172.21.32.42:9200, 172.21.32.39:9200, 172.21.32.38:9200, 172.21.32.36:9200, 172.21.32.37:9200, 172.21.32.43:9200, 172.21.32.40:9200, 172.21.32.41:9200]] 
	at org.apache.spark.TaskContextImpl.markTaskCompleted(TaskContextImpl.scala:105)
	at org.apache.spark.scheduler.Task.run(Task.scala:99)
	at org.apache.spark.executor.Executor$TaskRunner.run(Executor.scala:274)
	at java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1142)
	at java.util.concurrent.ThreadPoolExecutor$Worker.run(ThreadPoolExecutor.java:617)
	at java.lang.Thread.run(Thread.java:745)

Driver stacktrace:
	at org.apache.spark.sche

In [ ]:
requests.post('http://elasticsearchvis.service.consul:9200/case/_refresh')

In [ ]:
#requests.get('http://localhost:9200/test/_search?size=5').json()['hits']['hits']

In [ ]:
test_query = {
  "query": {
    "nested": {
      "path": "case",
      "score_mode": "sum",
      "query": {
        "function_score": {
          "query": {
            "bool": {
              "must": [
                {
                "terms": {
                  "case.project.project_id": [
                    "TCGA-ACC"
                  ]
                }
                }
              ]
            }
          }
        }
      }
    }
  }
}
      
len(requests.post('http://localhost:9200/test/_search', data=json.dumps(test_query)).json()['hits']['hits'])